In [1]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      2
  On-line CPU(s) list:       0,1
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.00GHz
    CPU family:              6
    Model:                   85
    Thread(s) per core:      2
    Core(s) per socket:      1
    Socket(s):               1
    Stepping:                3
    BogoMIPS:                4000.32
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2ap
                   

In [2]:
!nvidia-smi

Sat Apr 11 22:23:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [4]:
import cupy as cp

In [5]:
stream = cp.cuda.Stream(non_blocking=False)

In [6]:
with stream:
  rand_arr = cp.random.random(1000000)

In [7]:
stream.synchronize()

In [8]:
print(rand_arr)

[0.9536223  0.19947192 0.34842895 ... 0.55510616 0.51392922 0.69293297]


In [9]:
import numpy as np
import time

def compute_elements(num_elements: int = 1000, ex=np):
    rd = ex.random.RandomState(88)
    a = rd.randint(1, num_elements, (num_elements, num_elements))
    y = rd.randint(1, num_elements, (num_elements))
    res = ex.linalg.solve(a, y)
    return res

In [10]:
import cupy as cp
import time

stream = cp.cuda.Stream(non_blocking=True)

sizes = [10, 50, 100, 250, 500, 750, 1000, 2500, 5000, 7500, 10000, 15000]

timed_results = []
with stream:
    for n_elems in sizes:
        start_time = time.time()
        res = compute_elements(n_elems, ex=cp)
        stream.synchronize()
        end_time = time.time()
        print(f"{n_elems} elements res: {res.shape}")
        timed_results.append(end_time - start_time)

print("Timed Results:", timed_results)

10 elements res: (10,)
50 elements res: (50,)
100 elements res: (100,)
250 elements res: (250,)
500 elements res: (500,)
750 elements res: (750,)
1000 elements res: (1000,)
2500 elements res: (2500,)
5000 elements res: (5000,)
7500 elements res: (7500,)
10000 elements res: (10000,)
15000 elements res: (15000,)
Timed Results: [1.7014760971069336, 0.002161264419555664, 0.0018193721771240234, 0.003624439239501953, 0.009250402450561523, 0.10190820693969727, 0.02737712860107422, 0.1511223316192627, 0.8485808372497559, 1.2871661186218262, 2.9415283203125, 9.54045033454895]


In [12]:
import cupy as cp
import time
import csv
import statistics

def compute_elements(num_elements: int = 1000, ex=cp):
    rd = ex.random.RandomState(88)
    a = rd.randint(1, num_elements, (num_elements, num_elements))
    y = rd.randint(1, num_elements, (num_elements))
    res = ex.linalg.solve(a, y)
    return res


sizes = [10, 50, 100, 250, 500, 750, 1000, 2500, 5000, 7500, 10000, 15000]

with open("gpu_results.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["elements", "run", "median"])

    stream = cp.cuda.Stream(non_blocking=True)

    for n_elems in sizes:
        runtimes = []

        for run in range(3):
            with stream:
                start_time = time.time()
                res = compute_elements(n_elems, ex=cp)
                stream.synchronize()
                end_time = time.time()

            runtime = end_time - start_time
            runtimes.append(runtime)

            writer.writerow([n_elems, runtime, ""])

        median_val = statistics.median(runtimes)

        writer.writerow([n_elems, "", median_val])

        print(f"{n_elems} elements | GPU median={median_val}")

10 elements | GPU median=0.001226186752319336
50 elements | GPU median=0.0013818740844726562
100 elements | GPU median=0.0016903877258300781
250 elements | GPU median=0.0033173561096191406
500 elements | GPU median=0.008789539337158203
750 elements | GPU median=0.017589807510375977
1000 elements | GPU median=0.026728391647338867
2500 elements | GPU median=0.09488177299499512
5000 elements | GPU median=0.42077088356018066
7500 elements | GPU median=1.288167953491211
10000 elements | GPU median=2.9679551124572754
15000 elements | GPU median=9.825241327285767
